# Neural World Model

Predict top-down ball motion from four video frames and known impulses. The default **smoke** run verifies the full pipeline with undertrained models; use `full` for research experiments. The package supplies all simulation, training, evaluation, and inference logic.

Locally: install the project, then open this notebook from `notebooks/`. In Colab: select a GPU runtime and upload the source bundle created by `python tools/make_colab_bundle.py`. Add `--include-artifacts --config configs/full.json` to carry a resumable experiment.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile

IN_COLAB = 'google.colab' in sys.modules
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/nwm_project')]
project = next((p for p in candidates if (p / 'nwm').is_dir()), None)
if project is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    archive_path = next((Path(name) for name in uploaded if name.endswith('.zip')), None)
    if archive_path is None:
        raise ValueError('Upload nwm_colab.zip from tools/make_colab_bundle.py')
    project = Path('/content/nwm_project')
    project.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            if not (project / member.filename).resolve().is_relative_to(project.resolve()):
                raise ValueError('Archive path leaves the project directory')
        archive.extractall(project)
if project is None:
    raise FileNotFoundError('Open this notebook inside the extracted source project')
os.chdir(project)
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.'])
print('Project:', Path.cwd())


## Choose an experiment

`smoke` uses 32 training episodes and a few epochs. `full` uses 8,000 training episodes and all three dynamics seeds. Keep checkpoints and data between cloud sessions; runtime storage is temporary. The device check runs actual forward/backward kernels.


In [ ]:
from nwm.config import load_config
from nwm.training import device_check

CONFIG_NAME = 'smoke'  # change to 'full' for the full experiment
DEVICE = 'auto'
RUN_TRAINING = True
config = load_config(f'configs/{CONFIG_NAME}.json')
print(device_check(DEVICE))
print('Data:', config['data']['root'])
print('Results:', config['outputs']['root'])


In [ ]:
from nwm.data import generate_dataset
from nwm.training import train

# Compatible generation is idempotent; resume keeps the last complete epoch.
generate_dataset(config)
if RUN_TRAINING:
    train(config, device=DEVICE, resume=True)


## Compare predictors and uncertainty

Every model receives the same visual state estimate and action sequence. Separate true-initial-state results isolate dynamics error. OOD conditions change physics or speed; they are never used to tune weights or calibrate uncertainty. Calibration targets marginal per-object coverage at each horizon, not whole-trajectory or OOD guarantees.


In [ ]:
from nwm.evaluation import evaluate
from nwm.demo import make_demo

results = evaluate(config, device=DEVICE)
demo = make_demo(config, device=DEVICE)
print('Evaluation:', results)
print('Videos:', demo)


In [ ]:
import json
from IPython.display import display, Image, Video

report = json.loads((results / 'metrics.json').read_text())
for row in report['records']:
    if row['split'] == 'test' and row['initialization'] == 'estimated' and row['actions'] == 'recorded':
        print(row['model'], row['seed'], '50-frame position RMSE (px):',
              round(row['horizons']['50']['position_rmse_px'], 3))
print('Uncertainty:', report['uncertainty'][0])
display(Image(filename=str(results / 'error_vs_horizon.png')))
manifest = json.loads((demo / 'manifest.json').read_text())
for key in ('decoded_video', 'state_video'):
    path = Path(manifest[key])
    display(Video(str(path), embed=True) if path.suffix == '.mp4' else Image(filename=str(path)))


## Inference from video and actions

The predictor accepts no future states or collision labels. In the example, the held-out simulator supplies a video and a recorded future action trace. Ground truth is available to evaluation code, but is never passed to `predict`.


In [ ]:
import torch
from nwm.data import EpisodeDataset
from nwm.training import load_artifacts
from nwm.predict import EnsemblePredictor

artifacts = load_artifacts(config, DEVICE)
predictor = EnsemblePredictor(artifacts['estimator'], artifacts['decoder'],
    [model for seed, model in artifacts['hybrid']], artifacts['device'])
sample = EpisodeDataset(config['data']['root'], 'test', horizon=50)[0]
prediction = predictor.predict(sample['context_frames'][None],
    sample['context_actions'][None], sample['future_actions'][None])
print({name: tuple(value.shape) for name, value in prediction.items()})


## Preserve a cloud run

Download the new bundle before ending a cloud session. It contains the selected dataset, checkpoints, and reports but excludes the virtual environment. Exact epoch-boundary continuation is tested on the same backend; numerical results may differ across GPU architectures.


In [ ]:
# Uncomment to package this experiment for download or another machine.
# subprocess.check_call([sys.executable, 'tools/make_colab_bundle.py',
#     '--config', f'configs/{CONFIG_NAME}.json', '--include-artifacts'])
# if IN_COLAB:
#     from google.colab import files
#     files.download('nwm_colab.zip')
